<a href="https://colab.research.google.com/github/gurudattamanpreet/Practice/blob/main/Tourism_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline as Skpipeline
from imblearn.pipeline import Pipeline as Imbpipeline
from sklearn.pipeline import Pipeline as Skpipeline
from imblearn.pipeline import Pipeline as Imbpipeline

In [2]:
url= 'https://github.com/gurudattamanpreet/datasets/raw/main/Tourism.xlsx'
df=pd.read_excel(url,sheet_name='Tourism')

In [3]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [4]:
df.drop(['CustomerID'],axis=1,inplace=True)

In [5]:
df.shape

(4888, 19)

In [6]:
s=df.select_dtypes('object').columns.tolist()
for i in s:
  print(df[i].value_counts(),'\n')

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64 

Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64 

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64 

ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64 

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64 

Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64 



In [7]:
df['Gender']=df['Gender'].replace('Fe Male','Female')

In [8]:
df['Gender'].value_counts()

,count
Gender,
Male,2916
Female,1972


In [9]:
df.select_dtypes('number').nunique()

,0
ProdTaken,2
Age,44
CityTier,3
DurationOfPitch,34
NumberOfPersonVisiting,5
NumberOfFollowups,6
PreferredPropertyStar,3
NumberOfTrips,12
Passport,2
PitchSatisfactionScore,5


In [10]:
df.corr(numeric_only=True)['ProdTaken'].sort_values(ascending=False)

,ProdTaken
ProdTaken,1.000000
Passport,0.260844
NumberOfFollowups,0.112171
PreferredPropertyStar,0.099577
CityTier,0.086852
DurationOfPitch,0.078257
PitchSatisfactionScore,0.051394
NumberOfTrips,0.018898
NumberOfPersonVisiting,0.009627
NumberOfChildrenVisiting,0.007421


In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ProdTaken,4888.0,0.188216,0.390925,0.0,0.0,0.0,0.0,1.0
Age,4662.0,37.622265,9.316387,18.0,31.0,36.0,44.0,61.0
CityTier,4888.0,1.654255,0.916583,1.0,1.0,1.0,3.0,3.0
DurationOfPitch,4637.0,15.490835,8.519643,5.0,9.0,13.0,20.0,127.0
NumberOfPersonVisiting,4888.0,2.905074,0.724891,1.0,2.0,3.0,3.0,5.0
NumberOfFollowups,4843.0,3.708445,1.002509,1.0,3.0,4.0,4.0,6.0
PreferredPropertyStar,4862.0,3.581037,0.798009,3.0,3.0,3.0,4.0,5.0
NumberOfTrips,4748.0,3.236521,1.849019,1.0,2.0,3.0,4.0,22.0
Passport,4888.0,0.290917,0.454232,0.0,0.0,0.0,1.0,1.0
PitchSatisfactionScore,4888.0,3.078151,1.365792,1.0,2.0,3.0,4.0,5.0


In [12]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(['ProdTaken'],axis=1),df['ProdTaken'],test_size=0.2,random_state=1,stratify=df['ProdTaken'])

In [13]:
X_test.shape

(978, 18)

In [14]:
X_train.shape

(3910, 18)

    df.describe(include='object').T

    plt.rcParams ['figure.figsize'] = 80,50
    model = pipe.named_steps['model']
    plot_tree (model)

In [15]:
num_cols=X_train.select_dtypes('number').columns.tolist()
cat_cols=X_train.select_dtypes('object').columns.tolist()

num_pipeline=Skpipeline([('impute',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipeline=Skpipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False))])

preprocessor=ColumnTransformer([('num',num_pipeline,num_cols),('cat',cat_pipeline,cat_cols)],remainder='passthrough')

pipe = Imbpipeline([
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', DecisionTreeClassifier(random_state=42))
])

param_grid = {
    'model__criterion': ['gini', 'entropy'],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_leaf_nodes': [10, 20, 30, None]
}

In [16]:
grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('num',
                                                                         Pipeline(steps=[('impute',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age',
                                                                          'CityTier',
                                                                          'DurationOfPitch',
                                                                          'NumberOfPersonVisiting',
                                                                          'NumberOfFollowups',
                                                                          'PreferredPropertyStar',
                                                                          'NumberOfTrips',
                                                                          'Passport',
                                                                          'PitchS...
                                                                          'ProductPitched',
                                                                          'MaritalStatus',
                                                                          'Designation'])])),
                                       ('smote', SMOTE(random_state=42)),
                                       ('model',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__criterion': ['gini', 'entropy'],
                         'model__max_depth': [5, 10, 15],
                         'model__max_leaf_nodes': [10, 20, 30, None],
                         'model__min_samples_leaf': [1, 2, 4],
                         'model__min_samples_split': [2, 5, 10]},
             scoring='accuracy', verbose=2)

In [17]:
# print("Best Params:", grid_search.best_params_)
# print("Best CV Score:", grid_search.best_score_)

In [18]:
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test)
y_pred_train = best_model.predict(X_train)

print(accuracy_score(y_test, y_pred_test))
print(accuracy_score(y_train,y_pred_train))
print(classification_report(y_test, y_pred_test))
print(classification_report(y_train,y_pred_train))

0.8752556237218814
0.9856777493606138
              precision    recall  f1-score   support

           0       0.92      0.93      0.92       794
           1       0.68      0.64      0.66       184

    accuracy                           0.88       978
   macro avg       0.80      0.79      0.79       978
weighted avg       0.87      0.88      0.87       978

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3174
           1       0.97      0.96      0.96       736

    accuracy                           0.99      3910
   macro avg       0.98      0.98      0.98      3910
weighted avg       0.99      0.99      0.99      3910



In [19]:
from sklearn.ensemble import RandomForestClassifier
print(RandomForestClassifier().get_params().keys())

dict_keys(['bootstrap', 'ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'max_samples', 'min_impurity_decrease', 'min_samples_leaf', 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'n_estimators', 'n_jobs', 'oob_score', 'random_state', 'verbose', 'warm_start'])
